In [ ]:
from pathlib import Path

import pandas as pd

datenordner = Path(r"C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\Daten OULAD")

info = pd.read_csv(datenordner / "studentInfo.csv")
registrierung = pd.read_csv(datenordner / "studentRegistration.csv")

print("studentInfo:", info.shape)
print("studentRegistration:", registrierung.shape)
display(info.head())
display(registrierung.head())


studentInfo: (32593, 12)
studentRegistration: (32593, 5)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159,?
1,AAA,2013J,28400,-53,?
2,AAA,2013J,30268,-92,12
3,AAA,2013J,31604,-52,?
4,AAA,2013J,32885,-176,?


In [2]:
print(info["final_result"].value_counts(dropna=False))
print()
print(info["final_result"].value_counts(normalize=True, dropna=False).round(3))

final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

final_result
Pass           0.379
Withdrawn      0.312
Fail           0.216
Distinction    0.093
Name: proportion, dtype: float64


In [3]:
# Originalspalte behalten und eine numerische Version anlegen
registrierung["abmeldetag"] = pd.to_numeric(
    registrierung["date_unregistration"], errors="coerce"
)

print("Datentyp vorher:", registrierung["date_unregistration"].dtype)
print("Datentyp nachher:", registrierung["abmeldetag"].dtype)

print("\nWerte, die keine Zahl sind:")
print(
    registrierung.loc[
        registrierung["date_unregistration"].notna()
        & registrierung["abmeldetag"].isna(),
        "date_unregistration"
    ].value_counts().head(10)
)

print("\nAbmeldungen bis einschließlich Tag 14:",
      registrierung["abmeldetag"].le(14).sum())
print("Abmeldungen nach Tag 14:",
      registrierung["abmeldetag"].gt(14).sum())

Datentyp vorher: str
Datentyp nachher: float64

Werte, die keine Zahl sind:
date_unregistration
?    22521
Name: count, dtype: int64

Abmeldungen bis einschließlich Tag 14: 4474
Abmeldungen nach Tag 14: 5598


In [4]:
registrierung["abmeldetag"] = pd.to_numeric(
    registrierung["date_unregistration"].replace("?", pd.NA),
    errors="coerce"
)

print(registrierung["abmeldetag"].describe())
print()
print("Ohne Abmeldedatum:", registrierung["abmeldetag"].isna().sum())
print("Abmeldungen bis einschließlich Tag 14:",
      registrierung["abmeldetag"].le(14).sum())
print("Abmeldungen nach Tag 14:",
      registrierung["abmeldetag"].gt(14).sum())

count    10072.000000
mean        49.757645
std         82.460890
min       -365.000000
25%         -2.000000
50%         27.000000
75%        109.000000
max        444.000000
Name: abmeldetag, dtype: float64

Ohne Abmeldedatum: 22521
Abmeldungen bis einschließlich Tag 14: 4474
Abmeldungen nach Tag 14: 5598


In [5]:
schluessel = ["code_module", "code_presentation", "id_student"]

print("Doppelte Schlüssel in studentInfo:",
      info.duplicated(schluessel).sum())
print("Doppelte Schlüssel in studentRegistration:",
      registrierung.duplicated(schluessel).sum())

Doppelte Schlüssel in studentInfo: 0
Doppelte Schlüssel in studentRegistration: 0


In [6]:
# Ein Kurseintrag wird durch Modul, Kursdurchlauf und Person identifiziert.
schluessel = ["code_module", "code_presentation", "id_student"]

# Kursergebnis und Registrierungsdaten zusammenführen.
# validate prüft, dass auf beiden Seiten jeder Schlüssel höchstens einmal vorkommt.
# indicator zeigt uns, ob zu jedem Eintrag eine passende Zeile gefunden wurde.
basis = info.merge(
    registrierung,
    on=schluessel,
    how="left",
    validate="one_to_one",
    indicator=True
)

# "?" und andere nicht numerische Einträge werden zu fehlenden Werten.
# Die ursprünglichen Spalten bleiben erhalten.
basis["abmeldetag"] = pd.to_numeric(
    basis["date_unregistration"], errors="coerce"
)
basis["anmeldetag"] = pd.to_numeric(
    basis["date_registration"], errors="coerce"
)

# Prüfen, ob die Verbindung vollständig war und ob
# Kursergebnis und vorhandenes Abmeldedatum zusammenpassen.
print("Verbindung der Tabellen:")
print(basis["_merge"].value_counts())

print("\nKursergebnis × Abmeldedatum vorhanden:")
print(pd.crosstab(basis["final_result"], basis["abmeldetag"].notna()))

print("\nAnmeldungen nach Tag 14:", basis["anmeldetag"].gt(14).sum())
print("Ohne lesbaren Anmeldetag:", basis["anmeldetag"].isna().sum())

Verbindung der Tabellen:
_merge
both          32593
left_only         0
right_only        0
Name: count, dtype: int64

Kursergebnis × Abmeldedatum vorhanden:
abmeldetag    False  True 
final_result              
Distinction    3024      0
Fail           7043      9
Pass          12361      0
Withdrawn        93  10063

Anmeldungen nach Tag 14: 58
Ohne lesbaren Anmeldetag: 45


In [7]:
# Bedingungen für die drei auffälligen Fallgruppen festlegen.
withdrawn_ohne_datum = (
    basis["final_result"].eq("Withdrawn")
    & basis["abmeldetag"].isna()
)

fail_mit_datum = (
    basis["final_result"].eq("Fail")
    & basis["abmeldetag"].notna()
)

anmeldung_ohne_datum = basis["anmeldetag"].isna()

# Die ursprünglichen Einträge anzeigen: Steht dort z. B. "?"
print("Withdrawn ohne Abmeldetag – Originalwerte:")
print(basis.loc[withdrawn_ohne_datum, "date_unregistration"].value_counts(dropna=False))

print("\nFail mit Abmeldetag – Tage:")
print(basis.loc[fail_mit_datum, "abmeldetag"].sort_values().to_list())

print("\nAnmeldetag fehlt – Originalwerte:")
print(basis.loc[anmeldung_ohne_datum, "date_registration"].value_counts(dropna=False))

Withdrawn ohne Abmeldetag – Originalwerte:
date_unregistration
?    93
Name: count, dtype: int64

Fail mit Abmeldetag – Tage:
[-7.0, -4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 166.0]

Anmeldetag fehlt – Originalwerte:
date_registration
?    45
Name: count, dtype: int64


In [8]:
# Widersprüchliche oder zeitlich nicht einordenbare Fälle markieren.
unklarer_abbruch = (
    basis["final_result"].eq("Withdrawn")
    & basis["abmeldetag"].isna()
)
widerspruch = (
    basis["final_result"].ne("Withdrawn")
    & basis["abmeldetag"].notna()
)
unklare_anmeldung = basis["anmeldetag"].isna()

# Nur für diesen ersten Zeitvergleich eindeutig einordenbare Fälle verwenden.
vergleichsdaten = basis.loc[
    ~(unklarer_abbruch | widerspruch | unklare_anmeldung)
].copy()

ergebnisse = []

for tag in [7, 14, 28]:
    # Die Person war spätestens an diesem Tag angemeldet ...
    bereits_angemeldet = vergleichsdaten["anmeldetag"].le(tag)

    # ... und bis einschließlich dieses Tages nicht abgemeldet.
    noch_dabei = (
        vergleichsdaten["abmeldetag"].isna()
        | vergleichsdaten["abmeldetag"].gt(tag)
    )

    gruppe = vergleichsdaten.loc[bereits_angemeldet & noch_dabei]

    # Von dieser Gruppe: Wer meldete sich erst nach dem Stichtag ab?
    spaetere_abbrueche = gruppe["abmeldetag"].gt(tag).sum()

    ergebnisse.append({
        "Stichtag": tag,
        "Noch dabei": len(gruppe),
        "Spätere Abmeldungen": spaetere_abbrueche,
        "Anteil späterer Abmeldungen": round(
            spaetere_abbrueche / len(gruppe) * 100, 1
        ),
    })

display(pd.DataFrame(ergebnisse))

,Stichtag,Noch dabei,Spätere Abmeldungen,Anteil späterer Abmeldungen
0,7,28983,6632,22.9
1,14,27961,5580,20.0
2,28,27421,5012,18.3


In [9]:
# Aufgabenbeschreibung: Welche Aufgaben gibt es in welchem Kurs?
aufgaben = pd.read_csv(datenordner / "assessments.csv")

# Abgaben und Bewertungen der Lernenden einlesen.
abgaben = pd.read_csv(datenordner / "studentAssessment.csv")

# Datums- und Punktespalten sicher in Zahlen umwandeln.
abgaben["einreichtag"] = pd.to_numeric(
    abgaben["date_submitted"], errors="coerce"
)
abgaben["punkte"] = pd.to_numeric(
    abgaben["score"], errors="coerce"
)

# Jeder Abgabe ihren Kursdurchlauf zuordnen.
abgaben_mit_kurs = abgaben.merge(
    aufgaben[["id_assessment", "code_module", "code_presentation"]],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

# Ein Kurseintrag besteht aus Person, Modul und Kursdurchlauf.
schluessel = ["code_module", "code_presentation", "id_student"]

for tag in [7, 14, 28]:
    # Nur Abgaben berücksichtigen, die am Stichtag schon erfolgt waren.
    bis_tag = abgaben_mit_kurs[
        abgaben_mit_kurs["einreichtag"].le(tag)
    ]

    print(
        f"Bis Tag {tag}: "
        f"{len(bis_tag)} Abgaben, "
        f"{bis_tag[schluessel].drop_duplicates().shape[0]} "
        "Kurseinträge mit mindestens einer Abgabe"
    )

print("\nAbgaben ohne lesbaren Einreichtag:",
      abgaben_mit_kurs["einreichtag"].isna().sum())
print("Abgaben ohne lesbare Punktzahl:",
      abgaben_mit_kurs["punkte"].isna().sum())

Bis Tag 7: 2909 Abgaben, 1416 Kurseinträge mit mindestens einer Abgabe
Bis Tag 14: 5587 Abgaben, 3893 Kurseinträge mit mindestens einer Abgabe
Bis Tag 28: 25614 Abgaben, 20832 Kurseinträge mit mindestens einer Abgabe

Abgaben ohne lesbaren Einreichtag: 0
Abgaben ohne lesbare Punktzahl: 173


In [10]:
# Die große Aktivitätsdatei in Blöcken lesen, statt alles
# gleichzeitig im Arbeitsspeicher zu halten.
datei = datenordner / "studentVle.csv"
stichtage = [7, 14, 28]
aktive_kurse = {tag: set() for tag in stichtage}

spalten = [
    "code_module", "code_presentation",
    "id_student", "date"
]

for block in pd.read_csv(
    datei,
    usecols=spalten,
    chunksize=250_000
):
    # Aktivitätstage als Zahlen behandeln.
    block["date"] = pd.to_numeric(block["date"], errors="coerce")

    for tag in stichtage:
        # Nur Aktivitäten vom Kursbeginn bis einschließlich Stichtag.
        fruehe_aktivitaet = block[
            block["date"].between(0, tag)
        ]

        # Eindeutige Kombinationen aus Kurs und Person sammeln.
        aktive_kurse[tag].update(
            fruehe_aktivitaet[
                ["code_module", "code_presentation", "id_student"]
            ].itertuples(index=False, name=None)
        )

# Die Aktivität mit den Gruppen aus dem vorherigen Zeitvergleich abgleichen.
for tag in stichtage:
    gruppe = vergleichsdaten[
        vergleichsdaten["anmeldetag"].le(tag)
        & (
            vergleichsdaten["abmeldetag"].isna()
            | vergleichsdaten["abmeldetag"].gt(tag)
        )
    ]

    kursschluessel = set(
        gruppe[
            ["code_module", "code_presentation", "id_student"]
        ].itertuples(index=False, name=None)
    )

    anzahl_aktiv = len(kursschluessel & aktive_kurse[tag])

    print(
        f"Tag {tag}: {anzahl_aktiv} von {len(gruppe)} "
        f"noch angemeldeten Kurseinträgen mit VLE-Aktivität "
        f"seit Kursbeginn ({anzahl_aktiv / len(gruppe):.1%})"
    )

Tag 7: 23708 von 28983 noch angemeldeten Kurseinträgen mit VLE-Aktivität seit Kursbeginn (81.8%)
Tag 14: 25448 von 27961 noch angemeldeten Kurseinträgen mit VLE-Aktivität seit Kursbeginn (91.0%)
Tag 28: 26256 von 27421 noch angemeldeten Kurseinträgen mit VLE-Aktivität seit Kursbeginn (95.8%)


In [11]:
# Schlüssel für einen Kurseintrag.
schluessel = ["code_module", "code_presentation", "id_student"]

# Zwischenergebnisse der einzelnen Lese-Blöcke sammeln.
teile = []

for block in pd.read_csv(
    datenordner / "studentVle.csv",
    usecols=schluessel + ["date", "sum_click"],
    chunksize=250_000
):
    # Datums- und Klickspalten als Zahlen einlesen.
    block["date"] = pd.to_numeric(block["date"], errors="coerce")
    block["sum_click"] = pd.to_numeric(
        block["sum_click"], errors="coerce"
    )

    # Nur Aktivitäten ab Kursbeginn bis einschließlich Tag 14 behalten.
    block = block[block["date"].between(0, 14)].copy()

    # Klicks den beiden Zeitfenstern zuordnen.
    block["klicks_tag_0_bis_6"] = block["sum_click"].where(
        block["date"].between(0, 6), 0
    )
    block["klicks_tag_7_bis_14"] = block["sum_click"].where(
        block["date"].between(7, 14), 0
    )

    # Innerhalb jedes Blocks die Klicks je Kurseintrag addieren.
    teile.append(
        block.groupby(schluessel)[
            ["klicks_tag_0_bis_6", "klicks_tag_7_bis_14"]
        ].sum()
    )

# Die Teilergebnisse aller Blöcke je Kurseintrag zusammenzählen.
klicks = (
    pd.concat(teile)
    .groupby(level=schluessel)
    .sum()
    .reset_index()
)

print("Kurseinträge mit erfassten Klicks bis Tag 14:", len(klicks))
display(klicks.head())

Kurseinträge mit erfassten Klicks bis Tag 14: 26578


,code_module,code_presentation,id_student,klicks_tag_0_bis_6,klicks_tag_7_bis_14
0,AAA,2013J,11391,183,20
1,AAA,2013J,28400,175,66
2,AAA,2013J,30268,107,72
3,AAA,2013J,31604,22,158
4,AAA,2013J,32885,61,116


In [12]:
# Nur Kurseinträge auswählen, die an Tag 14 bereits angemeldet
# und noch nicht abgemeldet waren.
gruppe14 = vergleichsdaten.loc[
    vergleichsdaten["anmeldetag"].le(14)
    & (
        vergleichsdaten["abmeldetag"].isna()
        | vergleichsdaten["abmeldetag"].gt(14)
    )
].copy()

# Ziel für unsere Untersuchung:
# Hat sich der Kurseintrag NACH Tag 14 abgemeldet?
gruppe14["spaeter_abgemeldet"] = gruppe14["abmeldetag"].gt(14)

# Klickzahlen ergänzen; alle Kurseinträge aus gruppe14 behalten.
analyse14 = gruppe14.merge(
    klicks,
    on=schluessel,
    how="left",
    validate="one_to_one"
)

# Ohne VLE-Zeile im Zeitraum 0–14: null erfasste Klicks.
klickspalten = ["klicks_tag_0_bis_6", "klicks_tag_7_bis_14"]
analyse14[klickspalten] = analyse14[klickspalten].fillna(0)

# Für jede Gruppe Anzahl und typische Klickzahl (Median) anzeigen.
vergleich = analyse14.groupby("spaeter_abgemeldet").agg(
    kurseintraege=("id_student", "size"),
    median_klicks_tag_0_bis_6=("klicks_tag_0_bis_6", "median"),
    median_klicks_tag_7_bis_14=("klicks_tag_7_bis_14", "median"),
)

display(vergleich)

,kurseintraege,median_klicks_tag_0_bis_6,median_klicks_tag_7_bis_14
spaeter_abgemeldet,,,
False,22381,37.0,38.0
True,5580,22.0,23.0


Erster Befund: Die später abgemeldete Gruppe war schon in den ersten zwei Wochen im Median weniger aktiv. Das ist ein brauchbares Signal für unsere Projektidee. Ob wir damit einzelne spätere Abmeldungen zuverlässig erkennen können, zeigt erst die weitere Auswertung: Die Klickzahlen beider Gruppen können sich stark überschneiden.

In [13]:
# Klicks aus beiden Zeitfenstern zusammenzählen.
analyse14["klicks_bis_tag_14"] = (
    analyse14["klicks_tag_0_bis_6"]
    + analyse14["klicks_tag_7_bis_14"]
)

# Für beide Gruppen ausgewählte Perzentile anzeigen.
# Beispiel: 50 % ist der Median; bei 75 % liegen drei Viertel
# der jeweiligen Gruppe höchstens bei diesem Wert.
verteilung = analyse14.groupby("spaeter_abgemeldet")[
    "klicks_bis_tag_14"
].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).unstack()

verteilung.columns = ["10 %", "25 %", "Median", "75 %", "90 %"]
display(verteilung)

,10 %,25 %,Median,75 %,90 %
spaeter_abgemeldet,,,,,
False,2.0,26.0,86.0,201.0,367.0
True,0.0,13.0,57.0,145.0,274.0


In [14]:
# Alle Kurseinträge nach Klickzahl in ungefähr fünf gleich große
# Gruppen einteilen. Bei gleichen Werten können es weniger
# als fünf Gruppen werden; duplicates="drop" verhindert dann einen Fehler.
analyse14["aktivitaetsgruppe"] = pd.qcut(
    analyse14["klicks_bis_tag_14"],
    q=5,
    duplicates="drop"
)

# Je Aktivitätsgruppe: Anzahl der Kurseinträge und
# Anzahl der späteren Abmeldungen zählen.
quoten = analyse14.groupby(
    "aktivitaetsgruppe",
    observed=True
).agg(
    kurseintraege=("spaeter_abgemeldet", "size"),
    spaetere_abmeldungen=("spaeter_abgemeldet", "sum")
)

# Anteil der späteren Abmeldungen innerhalb jeder Gruppe berechnen.
quoten["abbruchquote_prozent"] = (
    100 * quoten["spaetere_abmeldungen"] / quoten["kurseintraege"]
).round(1)

display(quoten)
print(
    "Abbruchquote insgesamt:",
    round(100 * analyse14["spaeter_abgemeldet"].mean(), 1),
    "%"
)

,kurseintraege,spaetere_abmeldungen,abbruchquote_prozent
aktivitaetsgruppe,,,
"(-0.001, 15.0]",5746,1477,25.7
"(15.0, 54.0]",5519,1248,22.6
"(54.0, 114.0]",5529,1080,19.5
"(114.0, 228.0]",5590,989,17.7
"(228.0, 4733.0]",5577,786,14.1


Abbruchquote insgesamt: 20.0 %


In [15]:
kurs = ["code_module", "code_presentation"]

# Anzahl und spätere Abbruchquote je Kursdurchlauf berechnen.
kursuebersicht = analyse14.groupby(kurs).agg(
    kurseintraege=("id_student", "size"),
    abbruchquote=("spaeter_abgemeldet", "mean")
)
kursuebersicht["abbruchquote"] *= 100

# Median der Klicks getrennt nach späterer Abmeldung berechnen.
mediane = (
    analyse14.groupby(kurs + ["spaeter_abgemeldet"])[
        "klicks_bis_tag_14"
    ]
    .median()
    .unstack()
    .rename(columns={
        False: "median_ohne_spaetere_abmeldung",
        True: "median_mit_spaeterer_abmeldung"
    })
)

# Beide Ergebnisse nebeneinander anzeigen.
display(kursuebersicht.join(mediane).round(1))

kurseintraege  abbruchquote  \
code_module code_presentation                                
AAA         2013J                        370          13.2   
            2014J                        350          14.6   
BBB         2013B                       1605          21.6   
            2013J                       1850          14.2   
            2014B                       1288          13.1   
            2014J                       1779          14.1   
CCC         2014B                       1591          34.8   
            2014J                       2050          30.8   
DDD         2013B                       1194          27.4   
            2013J                       1675          25.3   
            2014B                       1048          29.9   
            2014J                       1463          21.0   
EEE         2013J                        932          13.3   
            2014B                        613          15.0   
            2014J                       1023          14.0   
FFF         2013B                       1519          20.9   
            2013J                       1997          19.6   
            2014B                       1308          20.8   
            2014J                       1862          19.0   
GGG         2013J                        925           4.2   
            2014B                        802           8.6   
            2014J                        717          13.2   

                               median_ohne_spaetere_abmeldung  \
code_module code_presentation                                   
AAA         2013J                                       155.0   
            2014J                                       110.0   
BBB         2013B                                        71.0   
            2013J                                        46.0   
            2014B                                        48.0   
            2014J                                        54.0   
CCC         2014B                                        94.0   
            2014J                                       104.5   
DDD         2013B                                        96.0   
            2013J                                        85.0   
            2014B                                        87.0   
            2014J                                        82.0   
EEE         2013J                                       132.5   
            2014B                                       104.0   
            2014J                                       128.0   
FFF         2013B                                       201.0   
            2013J                                       194.0   
            2014B                                       166.0   
            2014J                                       206.0   
GGG         2013J                                        14.5   
            2014B                                        20.0   
            2014J                                        30.0   

                               median_mit_spaeterer_abmeldung  
code_module code_presentation                                  
AAA         2013J                                       140.0  
            2014J                                       120.0  
BBB         2013B                                        38.0  
            2013J                                        30.0  
            2014B                                        35.0  
            2014J                                        41.5  
CCC         2014B                                        50.0  
            2014J                                        50.0  
DDD         2013B                                        58.0  
            2013J                                        50.0  
            2014B                                        64.0  
            2014J                                        52.0  
EEE         2013J                                        53.0  
            2014B    

Das bestätigt, warum wir den Kurskontext berücksichtigen müssen. In FFF 2014J liegen die Mediane bei 206 versus 116 Klicks; in GGG 2013J bei 14,5 versus 12. Auch die spätere Abbruchquote schwankt stark, etwa von 4,2 % (GGG 2013J) bis 34,8 % (CCC 2014B). Ein fester Grenzwert wie „unter 15 Klicks“ hätte in diesen Kursen völlig unterschiedliche Bedeutungen.
In fast allen Durchläufen ist die später abgemeldete Gruppe im Median weniger aktiv. AAA 2014J ist eine Ausnahme (110 gegenüber 120). Der Zusammenhang ist also hilfreich, aber keine universelle Regel.

In [16]:
# Eindeutige Aktivitätstage aus jedem Lese-Block sammeln.
tage_teile = []

for block in pd.read_csv(
    datenordner / "studentVle.csv",
    usecols=schluessel + ["date"],
    chunksize=250_000
):
    # Aktivitätstag als Zahl behandeln und nur Tage 0 bis 14 behalten.
    block["date"] = pd.to_numeric(block["date"], errors="coerce")
    fruehe_tage = block.loc[
        block["date"].between(0, 14),
        schluessel + ["date"]
    ]

    # Mehrere Klick-Einträge derselben Person am selben Tag
    # zählen zunächst innerhalb dieses Blocks nur einmal.
    tage_teile.append(fruehe_tage.drop_duplicates())

# Auch zwischen den Blöcken können dieselben Person-Tage vorkommen.
# Deshalb nach dem Zusammenfügen noch einmal Duplikate entfernen.
eindeutige_tage = (
    pd.concat(tage_teile, ignore_index=True)
    .drop_duplicates()
)

# Verschiedene aktive Tage je Kurseintrag zählen.
aktive_tage = (
    eindeutige_tage.groupby(schluessel)
    .size()
    .rename("aktive_tage_bis_14")
    .reset_index()
)

# Das neue Merkmal zur bisherigen Tag-14-Gruppe hinzufügen.
analyse14 = analyse14.merge(
    aktive_tage,
    on=schluessel,
    how="left",
    validate="one_to_one"
)
analyse14["aktive_tage_bis_14"] = (
    analyse14["aktive_tage_bis_14"].fillna(0).astype(int)
)

# Typische Zahl aktiver Tage für beide Ergebnisgruppen vergleichen.
display(
    analyse14.groupby("spaeter_abgemeldet")[
        "aktive_tage_bis_14"
    ].describe()[["count", "25%", "50%", "75%"]]
)

,count,25%,50%,75%
spaeter_abgemeldet,,,,
False,22381.0,2.0,5.0,8.0
True,5580.0,2.0,4.0,7.0


In [17]:
# Aktive Tage in leicht verständliche Gruppen einteilen.
# Werte reichen hier von 0 bis höchstens 15 Tagen (Tag 0 bis 14).
analyse14["gruppe_aktive_tage"] = pd.cut(
    analyse14["aktive_tage_bis_14"],
    bins=[-1, 0, 2, 5, 10, 15],
    labels=["0 Tage", "1–2 Tage", "3–5 Tage", "6–10 Tage", "11–15 Tage"]
)

# Anzahl der Kurseinträge und späteren Abmeldungen je Gruppe zählen.
tagesquoten = analyse14.groupby(
    "gruppe_aktive_tage",
    observed=True
).agg(
    kurseintraege=("spaeter_abgemeldet", "size"),
    spaetere_abmeldungen=("spaeter_abgemeldet", "sum")
)

# Abbruchquote innerhalb jeder Gruppe berechnen.
tagesquoten["abbruchquote_prozent"] = (
    100 * tagesquoten["spaetere_abmeldungen"]
    / tagesquoten["kurseintraege"]
).round(1)

display(tagesquoten)

,kurseintraege,spaetere_abmeldungen,abbruchquote_prozent
gruppe_aktive_tage,,,
0 Tage,2513,684,27.2
1–2 Tage,5464,1235,22.6
3–5 Tage,7785,1634,21.0
6–10 Tage,8357,1506,18.0
11–15 Tage,3842,521,13.6


## Zwischenstand: Frühwarnhinweis an Tag 14

**Fragestellung:** Können Informationen, die bis zum Ende von Kurstag 14 vorliegen, Hinweise auf eine spätere Kursabmeldung geben?

**Untersuchungsgruppe:** Kurseinträge, die spätestens an Tag 14 angemeldet und zu diesem Zeitpunkt noch nicht abgemeldet waren. Fälle mit nicht einordenbarem Anmelde- oder Abmeldezeitpunkt sowie widersprüchlichem Ergebnis und Abmeldedatum wurden für diese erste Analyse ausgeschlossen.

**Erste Befunde:** Unter 27.961 Kurseinträgen gab es 5.580 spätere Abmeldungen (20,0 %). Bis Tag 14 hatten 91,0 % mindestens eine erfasste VLE-Aktivität. Die spätere Abbruchquote sank von 25,7 % in der Gruppe mit höchstens 15 Klicks auf 14,1 % in der Gruppe mit mehr als 228 Klicks. Nach aktiven Tagen sank sie von 27,2 % bei 0 Tagen auf 13,6 % bei 11–15 Tagen.

**Einordnung:** Die Gruppen überschneiden sich deutlich. Klicks und aktive Tage beschreiben Plattformnutzung, nicht unmittelbar Lernen, Motivation oder Unterstützungsbedarf. Die Ergebnisse zeigen einen Zusammenhang, aber noch keine geprüfte Vorhersageleistung und keinen Effekt einer möglichen Unterstützung.

**Nächster Schritt:** Einen reproduzierbaren Datensatz mit ausschließlich bis Tag 14 bekannten Merkmalen erstellen und eine einfache Regel mit einem Modell bei gleicher Anzahl von Hinweisen vergleichen.


In [18]:
# Merkmale, die am Ende von Tag 14 bekannt sein können.
merkmale = [
    "code_module",
    "code_presentation",
    "anmeldetag",
    "klicks_tag_0_bis_6",
    "klicks_tag_7_bis_14",
    "aktive_tage_bis_14",
]

# id_student nur als Kennung für die spätere Datenaufteilung behalten.
# spaeter_abgemeldet ist das Ergebnis, das wir vorhersagen wollen.
modellbasis = analyse14[
    ["id_student"] + merkmale + ["spaeter_abgemeldet"]
].copy()

print("Zeilen und Spalten:", modellbasis.shape)
print("\nFehlende Werte je Spalte:")
print(modellbasis.isna().sum())

print("\nZielvariable:")
print(modellbasis["spaeter_abgemeldet"].value_counts())

display(modellbasis.head())

Zeilen und Spalten: (27961, 8)

Fehlende Werte je Spalte:
id_student             0
code_module            0
code_presentation      0
anmeldetag             0
klicks_tag_0_bis_6     0
klicks_tag_7_bis_14    0
aktive_tage_bis_14     0
spaeter_abgemeldet     0
dtype: int64

Zielvariable:
spaeter_abgemeldet
False    22381
True      5580
Name: count, dtype: int64


,id_student,code_module,code_presentation,anmeldetag,klicks_tag_0_bis_6,klicks_tag_7_bis_14,aktive_tage_bis_14,spaeter_abgemeldet
0,11391,AAA,2013J,-159.0,183.0,20.0,5,False
1,28400,AAA,2013J,-53.0,175.0,66.0,7,False
2,31604,AAA,2013J,-52.0,22.0,158.0,8,False
3,32885,AAA,2013J,-176.0,61.0,116.0,9,False
4,38053,AAA,2013J,-110.0,135.0,156.0,13,False


In [19]:
# Zählen, wie oft jede Person in der Modellbasis vorkommt.
eintraege_pro_person = modellbasis["id_student"].value_counts()

print("Kurseinträge insgesamt:", len(modellbasis))
print("Verschiedene Personen:", eintraege_pro_person.size)
print(
    "Personen mit mehr als einem Kurseintrag:",
    eintraege_pro_person.gt(1).sum()
)
print(
    "Kurseinträge dieser mehrfach vertretenen Personen:",
    eintraege_pro_person[eintraege_pro_person.gt(1)].sum()
)

Kurseinträge insgesamt: 27961
Verschiedene Personen: 25144
Personen mit mehr als einem Kurseintrag: 2686
Kurseinträge dieser mehrfach vertretenen Personen: 5503


In [20]:
# Projektordner aus dem bekannten Datenpfad ableiten:
# .../PortfolioProjekt/Daten OULAD -> .../PortfolioProjekt
projektordner = datenordner.parent

# Ordner für erzeugte Analysedaten bei Bedarf anlegen.
zielordner = projektordner / "data" / "processed"
zielordner.mkdir(parents=True, exist_ok=True)

# Die geprüfte Tag-14-Modellbasis speichern.
modelldatei = zielordner / "oulad_tag14_modellbasis.csv"
modellbasis.to_csv(modelldatei, index=False)

print("Gespeichert:", modelldatei)
print("Kurseinträge:", len(modellbasis))

Gespeichert: C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_tag14_modellbasis.csv
Kurseinträge: 27961


In [21]:
wochen_spalten = [
    "aktive_tage_0_bis_6",
    "aktive_tage_7_bis_14",
]

# Jeden bereits eindeutigen Person-Tag einem Zeitfenster zuordnen.
tage_mit_fenster = eindeutige_tage.assign(
    fenster=eindeutige_tage["date"].le(6).map({
        True: wochen_spalten[0],
        False: wochen_spalten[1],
    })
)

# Verschiedene aktive Tage pro Kurseintrag und Zeitfenster zählen.
tage_pro_woche = (
    tage_mit_fenster
    .groupby(schluessel + ["fenster"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=wochen_spalten, fill_value=0)
    .reset_index()
)

# Falls die Zelle erneut ausgeführt wird, alte Versionen
# der beiden Spalten vor dem Verbinden entfernen.
analyse14 = analyse14.drop(
    columns=wochen_spalten, errors="ignore"
).merge(
    tage_pro_woche,
    on=schluessel,
    how="left",
    validate="one_to_one",
)

# Keine erfasste Aktivität bedeutet 0 aktive Tage.
analyse14[wochen_spalten] = (
    analyse14[wochen_spalten].fillna(0).astype(int)
)

# Kontrollieren: Beide Zeitfenster müssen zusammen
# genau die bisherige Gesamtzahl aktiver Tage ergeben.
assert (
    analyse14[wochen_spalten].sum(axis=1)
    == analyse14["aktive_tage_bis_14"]
).all()

print("Prüfung bestanden:", len(analyse14), "Kurseinträge")
display(
    analyse14.groupby("spaeter_abgemeldet")[
        wochen_spalten
    ].median()
)

Prüfung bestanden: 27961 Kurseinträge


,aktive_tage_0_bis_6,aktive_tage_7_bis_14
spaeter_abgemeldet,,
False,2.0,3.0
True,2.0,2.0


In [22]:
# Vorläufige Eingaben für Version 2 festlegen.
merkmale_v2 = [
    "code_module",
    "code_presentation",
    "anmeldetag",
    "klicks_tag_0_bis_6",
    "klicks_tag_7_bis_14",
    "aktive_tage_0_bis_6",
    "aktive_tage_7_bis_14",
]

# Kennung und späteres Ergebnis getrennt von den Eingaben behalten.
modellbasis_v2 = analyse14[
    ["id_student"] + merkmale_v2 + ["spaeter_abgemeldet"]
].copy()

# Prüfen, ob dieselben Kurseinträge in derselben Reihenfolge vorliegen.
assert modellbasis_v2["id_student"].equals(modellbasis["id_student"])
assert modellbasis_v2["spaeter_abgemeldet"].equals(
    modellbasis["spaeter_abgemeldet"]
)

# Eine neue Datei speichern; Version 1 bleibt erhalten.
datei_v2 = zielordner / "oulad_tag14_modellbasis_v2.csv"
modellbasis_v2.to_csv(datei_v2, index=False)

print("Gespeichert:", datei_v2)
print("Form:", modellbasis_v2.shape)
print("Fehlende Werte:", modellbasis_v2.isna().sum().sum())

Gespeichert: C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_tag14_modellbasis_v2.csv
Form: (27961, 9)
Fehlende Werte: 0


In [ ]:
from pathlib import Path

import pandas as pd

projektordner = Path(
    r"C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt"
)
datenordner = projektordner / "Daten OULAD"
zielordner = projektordner / "data" / "processed"
schluessel = ["code_module", "code_presentation", "id_student"]

if "eindeutige_tage" in globals():
    # Die bereits berechneten VLE-Tage aus diesem Notebook weiterverwenden.
    tage = eindeutige_tage[schluessel + ["date"]].copy()
else:
    # Falls der Kernel neu gestartet wurde: nur benötigte Spalten einlesen.
    teile = []

    for block in pd.read_csv(
        datenordner / "studentVle.csv",
        usecols=schluessel + ["date"],
        chunksize=500_000,
    ):
        block["date"] = pd.to_numeric(block["date"], errors="coerce")
        block = block.loc[block["date"].between(0, 14)]

        # Pro Person und Kurs reicht aus diesem Block der letzte Aktivitätstag.
        if not block.empty:
            teile.append(
                block.groupby(schluessel, as_index=False)["date"].max()
            )

    tage = pd.concat(teile, ignore_index=True)

# Ausschließlich Aktivität von Tag 0 bis einschließlich Tag 14 verwenden.
tage["date"] = pd.to_numeric(tage["date"], errors="coerce")
tage = tage.loc[tage["date"].between(0, 14)]

letzter_tag = (
    tage.groupby(schluessel, as_index=False)["date"]
    .max()
)

# V3 ergänzt die gespeicherte V1 um genau ein neues Merkmal.
basis_v1 = pd.read_csv(
    zielordner / "oulad_tag14_modellbasis.csv"
)
modellbasis_v3 = basis_v1.merge(
    letzter_tag,
    on=schluessel,
    how="left",
    validate="one_to_one",
)

ohne_vle_aktivitaet = modellbasis_v3["date"].isna().sum()

# 0 bedeutet: letzter Klick an Tag 14.
# 15 bedeutet: zwischen Tag 0 und 14 wurde kein Klick beobachtet.
modellbasis_v3["tage_seit_letzter_aktivitaet"] = (
    (14 - modellbasis_v3["date"])
    .fillna(15)
    .astype(int)
)
modellbasis_v3 = modellbasis_v3.drop(columns="date")

# Prüfen, dass beim Verbinden keine Kurseinträge verloren gingen.
assert len(modellbasis_v3) == len(basis_v1)
assert modellbasis_v3[schluessel + ["spaeter_abgemeldet"]].equals(
    basis_v1[schluessel + ["spaeter_abgemeldet"]]
)
assert modellbasis_v3["tage_seit_letzter_aktivitaet"].between(0, 15).all()
assert modellbasis_v3.isna().sum().sum() == 0

datei_v3 = zielordner / "oulad_tag14_modellbasis_v3.csv"
modellbasis_v3.to_csv(datei_v3, index=False)

print("Gespeichert:", datei_v3)
print("Form:", modellbasis_v3.shape)
print("Ohne VLE-Aktivität bis Tag 14:", ohne_vle_aktivitaet)
print(
    modellbasis_v3["tage_seit_letzter_aktivitaet"]
    .value_counts()
    .sort_index()
)

Gespeichert: C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_tag14_modellbasis_v3.csv
Form: (27961, 9)
Ohne VLE-Aktivität bis Tag 14: 2513
tage_seit_letzter_aktivitaet
0     8650
1     4459
2     3288
3     2330
4     1740
5     1284
6      767
7      448
8      423
9      453
10     474
11     395
12     387
13     180
14     170
15    2513
Name: count, dtype: int64


Alternative Zielgröße prüfen

In [ ]:
from pathlib import Path

import pandas as pd

projektordner = Path(
    r"C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt"
)
schluessel = ["code_module", "code_presentation", "id_student"]

# Die bestehende Tag-14-Gruppe mit dem späteren Kursergebnis verbinden.
basis = pd.read_csv(
    projektordner / "data" / "processed"
    / "oulad_tag14_modellbasis.csv"
)
info = pd.read_csv(
    projektordner / "Daten OULAD" / "studentInfo.csv",
    usecols=schluessel + ["final_result"],
)

zielpruefung = basis[
    schluessel + ["spaeter_abgemeldet"]
].merge(
    info,
    on=schluessel,
    how="left",
    validate="one_to_one",
)

print("Kursergebnisse der Tag-14-Gruppe:")
print(zielpruefung["final_result"].value_counts(dropna=False))

print("\nKursergebnis × bestätigte spätere Abmeldung:")
print(pd.crosstab(
    zielpruefung["final_result"],
    zielpruefung["spaeter_abgemeldet"],
    dropna=False,
))

# Bei diesen Fällen ist der Zeitpunkt der Abmeldung nicht bestätigt.
unklar = (
    zielpruefung["final_result"].eq("Withdrawn")
    & ~zielpruefung["spaeter_abgemeldet"]
)
print("\nWithdrawn ohne bestätigte spätere Abmeldung:", unklar.sum())

# Vorläufige Zieldefinition nur für Fälle mit eindeutigem Verlauf.
pruefgruppe = zielpruefung.loc[~unklar].copy()
pruefgruppe["abbruch_oder_fail"] = (
    pruefgruppe["spaeter_abgemeldet"]
    | pruefgruppe["final_result"].eq("Fail")
)

print("\nVorläufig auswertbare Fälle:", len(pruefgruppe))
print("Abbruch oder Fail:", pruefgruppe["abbruch_oder_fail"].sum())
print("Anteil:", round(pruefgruppe["abbruch_oder_fail"].mean(), 3))

Kursergebnisse der Tag-14-Gruppe:
final_result
Pass           12338
Fail            7021
Withdrawn       5580
Distinction     3022
Name: count, dtype: int64

Kursergebnis × bestätigte spätere Abmeldung:
spaeter_abgemeldet  False  True 
final_result                    
Distinction          3022      0
Fail                 7021      0
Pass                12338      0
Withdrawn               0   5580

Withdrawn ohne bestätigte spätere Abmeldung: 0

Vorläufig auswertbare Fälle: 27961
Abbruch oder Fail: 12601
Anteil: 0.451


Die Gesamtzahl aktiver Tage wird in Tag 0–6 und Tag 7–14 aufgeteilt. So kann das Modell erkennen, ob die regelmäßige Nutzung im zweiten Zeitfenster nachlässt. Alle Werte sind am Ende von Tag 14 bekannt.


In [ ]:
from pathlib import Path

import pandas as pd

projektordner = Path(
    r"C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt"
)
datenordner = projektordner / "Daten OULAD"
schluessel = ["code_module", "code_presentation", "id_student"]

info_28 = pd.read_csv(
    datenordner / "studentInfo.csv",
    usecols=schluessel + ["final_result"],
)
registrierung_28 = pd.read_csv(
    datenordner / "studentRegistration.csv",
    usecols=schluessel + [
        "date_registration",
        "date_unregistration",
    ],
)

tag28 = info_28.merge(
    registrierung_28,
    on=schluessel,
    how="inner",
    validate="one_to_one",
)

# Die Datumsangaben enthalten teilweise "?"; diese werden zu NaN.
tag28["anmeldetag"] = pd.to_numeric(
    tag28["date_registration"], errors="coerce"
)
tag28["abmeldetag"] = pd.to_numeric(
    tag28["date_unregistration"], errors="coerce"
)

# Stichtag: spätestens an Tag 28 angemeldet und bis dahin
# nicht nachweislich abgemeldet.
noch_dabei = (
    tag28["anmeldetag"].le(28)
    & (
        tag28["abmeldetag"].isna()
        | tag28["abmeldetag"].gt(28)
    )
)

# Bei Withdrawn ohne Abmeldedatum ist der Status an Tag 28 unklar.
unklar = (
    tag28["final_result"].eq("Withdrawn")
    & tag28["abmeldetag"].isna()
)

tag28 = tag28.loc[noch_dabei & ~unklar].copy()
tag28["abbruch_oder_fail"] = tag28["final_result"].isin(
    ["Withdrawn", "Fail"]
)

print("An Tag 28 noch dabei:", len(tag28))
print("\nSpätere Kursergebnisse:")
print(tag28["final_result"].value_counts())
print("\nAbbruch oder Fail:", tag28["abbruch_oder_fail"].sum())
print("Anteil:", round(tag28["abbruch_oder_fail"].mean(), 3))

An Tag 28 noch dabei: 27422

Spätere Kursergebnisse:
final_result
Pass           12355
Fail            7032
Withdrawn       5012
Distinction     3023
Name: count, dtype: int64

Abbruch oder Fail: 12044
Anteil: 0.439


In [26]:
assessments_28 = pd.read_csv(
    datenordner / "assessments.csv",
    usecols=[
        "id_assessment",
        "code_module",
        "code_presentation",
        "assessment_type",
        "date",
    ],
)
assessments_28["date"] = pd.to_numeric(
    assessments_28["date"], errors="coerce"
)

# Kurse mit mindestens einem regulären Assessment,
# dessen Abgabetermin bis einschließlich Tag 28 liegt.
faellige_kurse = (
    assessments_28.loc[
        assessments_28["assessment_type"].ne("Exam")
        & assessments_28["date"].between(0, 28),
        ["code_module", "code_presentation"],
    ]
    .drop_duplicates()
    .assign(assessment_faellig_bis28=True)
)

tag28 = tag28.merge(
    faellige_kurse,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one",
)
tag28["assessment_faellig_bis28"] = (
    tag28["assessment_faellig_bis28"]
    .fillna(False)
    .astype(bool)
)

abgaben_28 = pd.read_csv(
    datenordner / "studentAssessment.csv",
    usecols=[
        "id_assessment",
        "id_student",
        "date_submitted",
        "is_banked",
    ],
)
abgaben_28["date_submitted"] = pd.to_numeric(
    abgaben_28["date_submitted"], errors="coerce"
)
abgaben_28["is_banked"] = pd.to_numeric(
    abgaben_28["is_banked"], errors="coerce"
)

# Assessment-ID liefert den zugehörigen Kurs.
abgaben_28 = abgaben_28.merge(
    assessments_28[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "assessment_type",
        ]
    ],
    on="id_assessment",
    how="left",
    validate="many_to_one",
)

# Nur tatsächliche, nicht übertragene Abgaben von Tag 0 bis 28.
fruehe_abgaben = (
    abgaben_28.loc[
        abgaben_28["date_submitted"].between(0, 28)
        & abgaben_28["is_banked"].eq(0)
        & abgaben_28["assessment_type"].ne("Exam"),
        schluessel,
    ]
    .drop_duplicates()
    .assign(abgabe_bis28=True)
)

tag28 = tag28.merge(
    fruehe_abgaben,
    on=schluessel,
    how="left",
    validate="one_to_one",
)
tag28["abgabe_bis28"] = (
    tag28["abgabe_bis28"].fillna(False).astype(bool)
)

print("An Tag 28 noch dabei:", len(tag28))
print(
    "Mit mindestens einem fälligen Assessment:",
    tag28["assessment_faellig_bis28"].sum(),
)
print(
    "Mit mindestens einer tatsächlichen Abgabe:",
    tag28["abgabe_bis28"].sum(),
)
print("\nKursergebnis × frühe Abgabe:")
print(pd.crosstab(
    tag28["final_result"],
    tag28["abgabe_bis28"],
))

An Tag 28 noch dabei: 27422
Mit mindestens einem fälligen Assessment: 22441
Mit mindestens einer tatsächlichen Abgabe: 19829

Kursergebnis × frühe Abgabe:
abgabe_bis28  False  True 
final_result              
Distinction     715   2308
Fail           2681   4351
Pass           2647   9708
Withdrawn      1550   3462


In [27]:
# Trennt fehlende Abgaben danach, ob überhaupt schon etwas fällig war.
faelligkeit_und_abgabe = (
    tag28.groupby(
        ["assessment_faellig_bis28", "abgabe_bis28"],
        dropna=False,
    )["abbruch_oder_fail"]
    .agg(
        kurseintraege="size",
        abbruch_oder_fail="sum",
        anteil="mean",
    )
    .reset_index()
)

faelligkeit_und_abgabe["anteil_prozent"] = (
    faelligkeit_und_abgabe["anteil"] * 100
).round(1)

display(
    faelligkeit_und_abgabe[
        [
            "assessment_faellig_bis28",
            "abgabe_bis28",
            "kurseintraege",
            "abbruch_oder_fail",
            "anteil_prozent",
        ]
    ]
)

# Kleine Abweichung zur früheren Zählung (27.421) eingrenzen.
print("Anmeldung genau an Tag 28:", tag28["anmeldetag"].eq(28).sum())
print("Abmeldung genau an Tag 28:", tag28["abmeldetag"].eq(28).sum())

,assessment_faellig_bis28,abgabe_bis28,kurseintraege,abbruch_oder_fail,anteil_prozent
0,False,False,4719,1771,37.5
1,False,True,262,46,17.6
2,True,False,2874,2460,85.6
3,True,True,19567,7767,39.7


Anmeldung genau an Tag 28: 2
Abmeldung genau an Tag 28: 0


In [28]:
# Übertragene Assessment-Ergebnisse prüfen, deren Termin
# bis Tag 28 lag und deren Eintrag bis dahin vorlag.
banked = abgaben_28.loc[
    abgaben_28["is_banked"].eq(1)
    & abgaben_28["date_submitted"].le(28)
    & abgaben_28["assessment_type"].ne("Exam")
].merge(
    assessments_28[["id_assessment", "date"]].rename(
        columns={"date": "assessment_faellig_tag"}
    ),
    on="id_assessment",
    how="left",
    validate="many_to_one",
)

banked_bis28 = (
    banked.loc[
        banked["assessment_faellig_tag"].between(0, 28),
        schluessel,
    ]
    .drop_duplicates()
    .assign(banked_bis28=True)
)

pruefung_banked = tag28.merge(
    banked_bis28,
    on=schluessel,
    how="left",
    validate="one_to_one",
)
pruefung_banked["banked_bis28"] = (
    pruefung_banked["banked_bis28"].fillna(False).astype(bool)
)

gruppe_ohne_abgabe = (
    pruefung_banked["assessment_faellig_bis28"]
    & ~pruefung_banked["abgabe_bis28"]
)

print(
    "Übertragene Leistung in der Gruppe 'fällig, keine Abgabe':",
    pruefung_banked.loc[gruppe_ohne_abgabe, "banked_bis28"].sum(),
)
print(pd.crosstab(
    pruefung_banked.loc[gruppe_ohne_abgabe, "final_result"],
    pruefung_banked.loc[gruppe_ohne_abgabe, "banked_bis28"],
))

Übertragene Leistung in der Gruppe 'fällig, keine Abgabe': 458
banked_bis28  False  True 
final_result              
Distinction      20     17
Fail           1217    204
Pass            255    122
Withdrawn       924    115


### Modellbasis für Tag 28

Zielgruppe sind Personen, die am Ende von Tag 28 noch angemeldet sind.
Zielvariable ist eine spätere Abmeldung oder das Kursergebnis `Fail`.

Als frühe Merkmale nutzen wir VLE-Klicks bis Tag 28 sowie die Anzahl
fälliger, erfüllter und fehlender Assessments. Angerechnete Leistungen
zählen bei den bis Tag 28 fälligen Assessments als erfüllt.
Assessment-Noten werden nicht verwendet.

In [29]:
kurs = ["code_module", "code_presentation"]

# Wie viele Assessments waren je Kurs bis Tag 28 fällig?
faellige_assessments = assessments_28.loc[
    assessments_28["assessment_type"].ne("Exam")
    & assessments_28["date"].between(0, 28)
].copy()

anzahl_faellig = (
    faellige_assessments.groupby(kurs, as_index=False)["id_assessment"]
    .nunique()
    .rename(columns={"id_assessment": "assessments_faellig_bis28"})
)

# Den Abgaben ihren jeweiligen Abgabetermin zuordnen.
abgaben_mit_termin = abgaben_28.merge(
    assessments_28[["id_assessment", "date"]].rename(
        columns={"date": "abgabetermin"}
    ),
    on="id_assessment",
    how="left",
    validate="many_to_one",
)

# Eigene Abgaben ab Kursbeginn und angerechnete Leistungen
# berücksichtigen, sofern ihr Eintrag spätestens an Tag 28 vorliegt.
eigene_sichtbar = (
    abgaben_mit_termin["is_banked"].eq(0)
    & abgaben_mit_termin["date_submitted"].between(0, 28)
)
banked_sichtbar = (
    abgaben_mit_termin["is_banked"].eq(1)
    & abgaben_mit_termin["date_submitted"].le(28)
)

sichtbare_abgaben = abgaben_mit_termin.loc[
    abgaben_mit_termin["assessment_type"].ne("Exam")
    & (eigene_sichtbar | banked_sichtbar)
].copy()

# Eigene Abgaben zählen wir auch dann, wenn die Aufgabe erst später fällig ist.
anzahl_eigene = (
    sichtbare_abgaben.loc[sichtbare_abgaben["is_banked"].eq(0)]
    .groupby(schluessel)["id_assessment"]
    .nunique()
    .rename("abgaben_bis28")
    .reset_index()
)

# Als erfüllt zählt nur ein Assessment, das bis Tag 28 fällig war.
erfuellte_abgaben = sichtbare_abgaben.loc[
    sichtbare_abgaben["abgabetermin"].between(0, 28)
]

anzahl_erfuellt = (
    erfuellte_abgaben.groupby(schluessel)["id_assessment"]
    .nunique()
    .rename("assessments_erfuellt_bis28")
    .reset_index()
)

anzahl_banked = (
    erfuellte_abgaben.loc[erfuellte_abgaben["is_banked"].eq(1)]
    .groupby(schluessel)["id_assessment"]
    .nunique()
    .rename("banked_faellige_bis28")
    .reset_index()
)

In [30]:
# Nur bekannte Angaben am Stichtag und die Zielvariable übernehmen.
assessment_basis = tag28[
    schluessel + ["anmeldetag", "abbruch_oder_fail"]
].copy()

assessment_basis = assessment_basis.merge(
    anzahl_faellig,
    on=kurs,
    how="left",
    validate="many_to_one",
)

for tabelle in [anzahl_eigene, anzahl_erfuellt, anzahl_banked]:
    assessment_basis = assessment_basis.merge(
        tabelle,
        on=schluessel,
        how="left",
        validate="one_to_one",
    )

anzahlspalten = [
    "assessments_faellig_bis28",
    "abgaben_bis28",
    "assessments_erfuellt_bis28",
    "banked_faellige_bis28",
]
assessment_basis[anzahlspalten] = (
    assessment_basis[anzahlspalten].fillna(0).astype(int)
)

assessment_basis["assessments_fehlend_bis28"] = (
    assessment_basis["assessments_faellig_bis28"]
    - assessment_basis["assessments_erfuellt_bis28"]
)

assert len(assessment_basis) == len(tag28)
assert not assessment_basis[schluessel].duplicated().any()
assert assessment_basis["assessments_fehlend_bis28"].ge(0).all()

print("Form der Zwischenbasis:", assessment_basis.shape)
print(
    "Mit mindestens einem fälligen Assessment:",
    assessment_basis["assessments_faellig_bis28"].gt(0).sum(),
)
print(
    "Mit mindestens einer eigenen Abgabe:",
    assessment_basis["abgaben_bis28"].gt(0).sum(),
)

Form der Zwischenbasis: (27422, 10)
Mit mindestens einem fälligen Assessment: 22441
Mit mindestens einer eigenen Abgabe: 19829


In [31]:
klickspalten = [
    "klicks_tag_0_bis_6",
    "klicks_tag_7_bis_14",
    "klicks_tag_15_bis_28",
]
klick_teile = []

for block in pd.read_csv(
    datenordner / "studentVle.csv",
    usecols=schluessel + ["date", "sum_click"],
    chunksize=400_000,
):
    block["date"] = pd.to_numeric(block["date"], errors="coerce")
    block = block.loc[block["date"].between(0, 28)].copy()

    if block.empty:
        continue

    block["sum_click"] = pd.to_numeric(
        block["sum_click"], errors="coerce"
    )
    assert block["sum_click"].notna().all()

    # Jeder Klick wird genau einem der drei Zeitfenster zugeordnet.
    block["fenster"] = pd.cut(
        block["date"],
        bins=[-1, 6, 14, 28],
        labels=klickspalten,
    )

    klick_teile.append(
        block.groupby(
            schluessel + ["fenster"],
            observed=True,
        )["sum_click"]
        .sum()
        .unstack(fill_value=0)
    )

# Teilergebnisse der CSV-Blöcke je Kurseintrag addieren.
klicks_28 = (
    pd.concat(klick_teile)
    .groupby(level=schluessel)
    .sum()
    .reindex(columns=klickspalten, fill_value=0)
    .reset_index()
)

modellbasis_tag28 = assessment_basis.merge(
    klicks_28,
    on=schluessel,
    how="left",
    validate="one_to_one",
)
modellbasis_tag28[klickspalten] = (
    modellbasis_tag28[klickspalten].fillna(0).astype(int)
)

assert len(modellbasis_tag28) == len(tag28)
assert modellbasis_tag28.isna().sum().sum() == 0

zielordner = projektordner / "data" / "processed"
zielordner.mkdir(parents=True, exist_ok=True)
datei_tag28 = zielordner / "oulad_tag28_modellbasis.csv"
modellbasis_tag28.to_csv(datei_tag28, index=False)

print("Gespeichert:", datei_tag28)
print("Form:", modellbasis_tag28.shape)
print("Abbruch oder Fail:", modellbasis_tag28["abbruch_oder_fail"].sum())
print("Fehlende Werte:", modellbasis_tag28.isna().sum().sum())
display(modellbasis_tag28.head())

Gespeichert: C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_tag28_modellbasis.csv
Form: (27422, 13)
Abbruch oder Fail: 12044
Fehlende Werte: 0


,code_module,code_presentation,id_student,anmeldetag,abbruch_oder_fail,assessments_faellig_bis28,abgaben_bis28,assessments_erfuellt_bis28,banked_faellige_bis28,assessments_fehlend_bis28,klicks_tag_0_bis_6,klicks_tag_7_bis_14,klicks_tag_15_bis_28
0,AAA,2013J,11391,-159.0,False,1,1,1,0,0,183,20,100
1,AAA,2013J,28400,-53.0,False,1,1,1,0,0,175,66,162
2,AAA,2013J,31604,-52.0,False,1,1,1,0,0,22,158,145
3,AAA,2013J,32885,-176.0,False,1,1,1,0,0,61,116,95
4,AAA,2013J,38053,-110.0,False,1,1,1,0,0,135,156,159


Kursdauer und Interventionsfenster

In [ ]:
from pathlib import Path

import pandas as pd

# Projektordner finden, wenn das Notebook im Projektordner
# oder im Unterordner "notebooks" ausgeführt wird.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

kurse = pd.read_csv(projektordner / "Daten OULAD" / "courses.csv")

laengenspalte = (
    "module_presentation_length"
    if "module_presentation_length" in kurse.columns
    else "length"
)

kursdauer = (
    kurse[
        ["code_module", "code_presentation", laengenspalte]
    ]
    .rename(columns={laengenspalte: "dauer_tage"})
    .sort_values(["code_module", "code_presentation"])
)

# Tag 0 bis einschließlich Tag 28 sind 29 Kalendertage.
kursdauer["anteil_bis_tag28_prozent"] = (
    29 / kursdauer["dauer_tage"] * 100
).round(1)

display(kursdauer)
print("\nKürzester, mittlerer und längster Durchlauf:")
print(kursdauer["dauer_tage"].agg(["min", "median", "max"]))

,code_module,code_presentation,dauer_tage,anteil_bis_tag28_prozent
0,AAA,2013J,268,10.8
1,AAA,2014J,269,10.8
4,BBB,2013B,240,12.1
2,BBB,2013J,268,10.8
5,BBB,2014B,234,12.4
3,BBB,2014J,262,11.1
7,CCC,2014B,241,12.0
6,CCC,2014J,269,10.8
10,DDD,2013B,240,12.1
8,DDD,2013J,261,11.1



Kürzester, mittlerer und längster Durchlauf:
min       234.0
median    261.5
max       269.0
Name: dauer_tage, dtype: float64
